# oscNext L4 — pybdt ile uçtan uca

L3 `.i3` dosyalarından başlayıp eğitilmiş bir BDT'ye kadar tüm süreç.

**Motor: pybdt** (IceCube'un kendi AdaBoost kütüphanesi). Bu, teknik notun
(v00.074, bölüm 3.6.1 — LightGBM) resmi yönteminden bilinçli bir sapmadır;
gerekçe ve sonuçları için `CLAUDE.md`.

## Çalıştırma ortamı

Bu notebook'u başlatan Jupyter, pybdt'nin derlendiği build'in env-shell'i
içinden başlatılmış olmalı:

```bash
eval $(/cvmfs/icecube.opensciencegrid.org/py3-v4.4.2/setup.sh)
cd /data/user/$(whoami)/icetray_build/build && ./env-shell.sh
cd ~/l4 && python -m jupyter lab --no-browser --port=8896
```

## Bölümler

| # | Ne yapar | Tekrar çalıştırılır mı |
|---|---|---|
| 0 | Konfigürasyon + ortam kontrolü | her açılışta |
| 1 | L3 → L4 işleme (`process_L4.py`) | bir kez, uzun sürer |
| 2 | Booking doğrulaması — HDF5'te gerçekte ne var | bir kez |
| 3 | Feature registry — hangi kolon hangi değişken | değişken seçimi değişince |
| 4 | HDF5 → numpy dizileri | sık |
| 5 | Ağırlıklar | sık |
| 6 | pybdt DataSet + train/test ayrımı → `.ds` | sık |
| 7 | Eğitim (`pybdt_train.py`) | sık |
| 8 | Doğrulama — overtraining, dağılımlar | sık |
| 9 | Kesim seçimi | sık |
| 10 | Modeli frame'e uygulama | en son |

## 0. Konfigürasyon ve ortam kontrolü

Önce neyin mevcut olduğunu tespit ediyoruz. `pybdt` zorunlu; `tables`
(pytables) HDF5 okumak için zorunlu. `pandas` **kullanılmıyor** — IceTray
ortamında bulunmayabilir.

In [ ]:
import os, sys, glob, json, shlex, subprocess, time
import numpy as np

# --- zorunlu: pybdt ---
try:
    from pybdt import ml, util
    print("pybdt        OK  ", os.path.dirname(ml.__file__))
except ImportError as e:
    raise SystemExit(
        "pybdt import edilemedi (%s).\n"
        "Jupyter'i pybdt build'inin env-shell'inden baslatin -- bkz. README.\n"
        "NOT: dogru import 'import pybdt', 'from icecube import pybdt' DEGIL." % e)

# --- zorunlu: pytables ---
try:
    import tables
    print("tables       OK  ", tables.__version__)
except ImportError:
    raise SystemExit("pytables yok -- HDF5 okunamaz.")

# --- opsiyonel ---
for _n in ("matplotlib", "scipy", "simweights"):
    try:
        _m = __import__(_n)
        print("%-12s OK   %s" % (_n, getattr(_m, "__version__", "")))
    except ImportError:
        print("%-12s YOK  (%s)" % (_n, {
            "matplotlib": "grafik cizilemez, pybdt.validate calismaz",
            "scipy":      "pybdt.validate calismaz",
            "simweights": "CORSIKA agirligi yaklasik olur"}[_n]))

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

In [ ]:
# ---------------------------------------------------------------------------
# YOLLAR
# ---------------------------------------------------------------------------
# Scriptlerin (process_L4.py, pybdt_train.py) bulundugu dizin
L4_CODE_DIR = os.environ.get("OSCNEXT_L4_CODE", ".")

# Ciktilar.  HDF5 ONLARCA GB olabilir -- home dizininde kota varsa
# OSCNEXT_OUT_ROOT'u /data/user/$USER/... altina alin.
OUTPUT_ROOT = os.environ.get(
    "OSCNEXT_OUT_ROOT", os.path.join(os.path.abspath(L4_CODE_DIR), "L4_output"))

HDF_BASE  = os.path.join(OUTPUT_ROOT, "hdf5")     # process_L4.py ciktisi
DS_BASE   = os.path.join(OUTPUT_ROOT, "ds")       # pybdt DataSet dosyalari
MODEL_DIR = os.path.join(OUTPUT_ROOT, "models")   # .bdt + .validator + grafikler

for _d in (HDF_BASE, DS_BASE, MODEL_DIR):
    os.makedirs(_d, exist_ok=True)

PROCESS_PY = os.path.join(L4_CODE_DIR, "process_L4.py")
TRAIN_PY   = os.path.join(L4_CODE_DIR, "pybdt_train.py")

RNG_SEED = 12345
rng = np.random.default_rng(RNG_SEED)

_st = os.statvfs(OUTPUT_ROOT)
print("Cikti koku : %s" % OUTPUT_ROOT)
print("Bos disk   : %.1f GB" % (_st.f_bavail * _st.f_frsize / 1e9))
for _p in (PROCESS_PY, TRAIN_PY):
    print("%-14s %s" % (os.path.basename(_p), "var" if os.path.exists(_p) else "YOK!"))

## 1. L3 → L4 işleme

`process_L4.py`'yi her örnek için çalıştırır. Bu adım **uzun sürer** ve bir
kez yapılır — HDF5 dosyaları üretildikten sonra 2. bölümden devam edebilirsin.

`--apply-cut` **kullanılmıyor**: modeller eğitilmeden önce tüm olaylar book
edilmeli, yoksa eğitim setini kesmiş oluruz.

In [ ]:
GCD = "/cvmfs/icecube.opensciencegrid.org/data/GCD/GeoCalibDetectorStatus_2020.Run134142.Pass2_V0.i3.gz"

# pass3 uretimi.  ntau ve gercek dedektor verisi bu uretimde YOK.
SAMPLES = {
    "nue":     dict(l3="/data/ana/LE/oscNext/pass3/genie/level3/23800/*.i3.zst",
                    flags=["--mc", "--genie"], kind="signal"),
    "numu":    dict(l3="/data/ana/LE/oscNext/pass3/genie/level3/23799/*.i3.zst",
                    flags=["--mc", "--genie"], kind="signal"),
    "corsika": dict(l3="/data/ana/LE/oscNext/pass3/corsika/level3/23694/*.i3.zst",
                    flags=["--corsika"], kind="muon_bg"),
    "noise":   dict(l3="/data/ana/LE/oscNext/pass3/noise/level3/23813/*.i3.zst",
                    flags=["--noise"], kind="noise_bg"),
}

for name, cfg in SAMPLES.items():
    cfg["hdf5"] = os.path.join(HDF_BASE, name, "L4_%s.hdf5" % name)
    n = len(glob.glob(cfg["l3"]))
    cfg["n_l3_files"] = n
    print("%-8s %-10s %6d L3 dosyasi" % (name, cfg["kind"], n))

In [ ]:
def run_process(name, n_frames=0, log=True):
    """process_L4.py'yi bir ornek icin calistir.  n_frames>0 -> smoke test."""
    cfg = SAMPLES[name]
    out = cfg["hdf5"] if n_frames == 0 else cfg["hdf5"].replace(".hdf5", "_smoke.hdf5")
    os.makedirs(os.path.dirname(out), exist_ok=True)

    cmd = [sys.executable, PROCESS_PY,
           "--gcd", GCD, "--input", cfg["l3"],
           "--output-hdf5", out] + cfg["flags"]
    if n_frames:
        cmd += ["--n", str(n_frames)]

    print("$ " + " ".join(shlex.quote(c) for c in cmd))
    t0 = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True)
    dt = time.time() - t0

    if log:
        tail = (p.stdout or "").strip().split("\n")[-25:]
        print("\n".join(tail))
    if p.returncode != 0:
        print("\n--- HATA (returncode=%d) ---" % p.returncode)
        print((p.stderr or "").strip()[-3000:])
    else:
        sz = os.path.getsize(out) / 1e6 if os.path.exists(out) else 0
        print("\n-> %s  (%.1f MB, %.0f s)" % (out, sz, dt))
    return out if p.returncode == 0 else None

### Önce küçük bir smoke test

Tam üretime girmeden önce 200 frame ile dene — tray kuruluyor mu, tablolar
book ediliyor mu?

In [ ]:
smoke = run_process("nue", n_frames=200)

### Tam üretim

Smoke test temizse tüm örnekleri işle. Bu hücre saatler sürebilir — cobalt
paylaşılan bir makine, uzun işleri tercihen `condor` üzerinden çalıştır.

In [ ]:
# for name in SAMPLES:
#     print("=" * 70); print(name); print("=" * 70)
#     run_process(name)

## 2. Booking doğrulaması

HDF5'te **gerçekte** hangi tablolar ve kolonlar var? 3. bölümdeki feature
registry'yi buna göre hizalayacağız.

Sütun isimleri `simple_booker.py` (pytables fallback) ile `icecube.hdfwriter`
arasında farklılık gösterebilir — bu yüzden varsayım yapmak yerine bakıyoruz.

In [ ]:
def dump_tables(h5path, only=None, max_cols=40):
    """HDF5'teki tablolari ve kolonlarini listele."""
    if not os.path.exists(h5path):
        print("YOK:", h5path); return {}
    found = {}
    with tables.open_file(h5path, "r") as h5:
        for node in h5.walk_nodes("/", "Table"):
            cols = [c for c in node.colnames
                    if c not in ("Run", "Event", "SubEvent", "SubEventStream", "exists")]
            found[node.name] = (node.nrows, cols)
    for name in sorted(found):
        nrows, cols = found[name]
        if only and not any(o in name for o in only):
            continue
        print("%-45s %8d satir" % (name, nrows))
        for c in cols[:max_cols]:
            print("      %s" % c)
        if len(cols) > max_cols:
            print("      ... (+%d kolon)" % (len(cols) - max_cols))
    return found


TABLES = dump_tables(SAMPLES["nue"]["hdf5"].replace(".hdf5", "_smoke.hdf5"))

## 3. Feature registry

**Tek doğruluk kaynağı**: hangi BDT değişkeni, HDF5'te hangi tablonun hangi
kolonundan okunur.

Değişken listeleri teknik nottan (Tablo 11 = noise, Tablo 12 = muon).
`l4_classifier_module.py`'deki `FEATURE_MAP` ile **aynı** değişkenleri
tanımlar — orası frame anahtarlarını, burası HDF5 kolonlarını gösterir.
İkisinden biri değişirse diğeri de değişmeli, yoksa eğitim ve uygulama
farklı şeyleri okur ve model **sessizce** saçmalar.

In [ ]:
# BDT degisken adi -> (HDF5 tablosu, kolon)
L3V     = "IC2018_LE_L3_Vars"
HITSTAT = "SRTTWSplitInIcePulsesDCHitStatistics"
HITMULT = "SRTTWSplitInIcePulsesDCHitMultiplicity"

REGISTRY = {
    # --- noise BDT (Tablo 11) ---
    "NchCleaned":          (L3V, "NchCleaned"),
    "micro_count":         ("L4_micro_count", "STW_m3500p4000_DTW200"),
    "iLineFit_speed":      ("L4_iLineFitParams", "LFVel"),
    "fill_ratio":          ("L4_fill_ratio", "fill_ratio_from_mean"),
    "FullTimeLengthRatio": ("L4_FullTimeLengthRatio", "value"),

    # --- muon BDT (Tablo 12) ---
    "ICVetoHits":       (L3V, "ICVetoHits"),
    "RTVeto250Hits":    (L3V, "RTVeto250Hits"),
    "NAbove200Hits":    (L3V, "NAbove200Hits"),
    "VICH_nch":         ("L4_VICH_nch", "value"),
    "accumulated_time": ("L4_accumulated_time", "value"),
    "first_hlc_rho":    ("L4_first_hlc_rho", "value"),
    "cog_z":            (HITSTAT, "cog_z"),
    "z_sigma":          (HITSTAT, "z_sigma"),
    "z_travel":         (HITSTAT, "z_travel"),

    # --- aday / tarama icin ---
    "n_hit_doms":       (HITMULT, "n_hit_doms"),
    "C2HR6":            (L3V, "C2HR6"),
    "CausalVetoHits":   (L3V, "CausalVetoHits"),
    "VertexGuessZ":     (L3V, "VertexGuessZ"),
    "DCFiducialHits":   (L3V, "DCFiducialHits"),
}

NOISE_FEATURES = ["NchCleaned", "micro_count", "iLineFit_speed",
                  "fill_ratio", "FullTimeLengthRatio"]

MUON_FEATURES = ["ICVetoHits", "RTVeto250Hits", "NAbove200Hits", "VICH_nch",
                 "accumulated_time", "first_hlc_rho", "cog_z", "z_sigma",
                 "z_travel"]

# Agirlik hesabi icin gereken ek kolonlar (BDT girdisi DEGIL)
AUX = {
    "true_energy":   ("I3MCWeightDict", "PrimaryNeutrinoEnergy"),
    "OneWeight":     ("I3MCWeightDict", "OneWeight"),
    "NEvents":       ("I3MCWeightDict", "NEvents"),
    "pdg":           ("I3MCWeightDict", "PrimaryNeutrinoType"),
    "n_flux_events": ("L4_n_flux_events", "value"),
    "noise_weight":  ("noise_weight", "value"),
}


def check_registry(found, names):
    """Registry'deki kolonlar HDF5'te gercekten var mi?"""
    ok, bad = [], []
    for n in names:
        tbl, col = REGISTRY.get(n, AUX.get(n, (None, None)))
        if tbl in found and col in found[tbl][1]:
            ok.append(n)
        else:
            bad.append((n, tbl, col))
    print("bulundu: %d/%d" % (len(ok), len(names)))
    for n, tbl, col in bad:
        why = "tablo yok" if tbl not in found else "kolon yok"
        print("  [!] %-22s %s[%s]  -- %s" % (n, tbl, col, why))
    return bad


print("--- noise ---");  check_registry(TABLES, NOISE_FEATURES)
print("--- muon ---");   check_registry(TABLES, MUON_FEATURES)
print("--- agirlik (AUX) ---"); check_registry(TABLES, list(AUX))

## 4. HDF5 → numpy

Tablolar olay bazında `(Run, Event, SubEvent)` ile hizalanır.

`simple_booker.py` bir anahtarı yalnızca frame'de varsa yazar, dolayısıyla
tablolar **farklı uzunlukta olabilir**. Hepsi aynı uzunlukta ve aynı sıradaysa
hızlı yol kullanılır; değilse sözlük tabanlı birleştirme yapılır ve eksikler
NaN olur.

In [ ]:
def _ids(tbl):
    return (np.asarray(tbl.col("Run"), dtype=np.int64),
            np.asarray(tbl.col("Event"), dtype=np.int64),
            np.asarray(tbl.col("SubEvent"), dtype=np.int64))


def _load_one_file(path, wanted):
    """Tek HDF5 dosyasindan istenen degiskenleri oku -> {ad: dizi}."""
    need = {}
    for name in wanted:
        tbl, col = REGISTRY.get(name, AUX.get(name, (None, None)))
        if tbl:
            need.setdefault(tbl, []).append((name, col))

    with tables.open_file(path, "r") as h5:
        nodes = {n.name: n for n in h5.walk_nodes("/", "Table")}
        if "I3EventHeader" not in nodes:
            raise RuntimeError("%s: I3EventHeader tablosu yok" % path)

        ref = nodes["I3EventHeader"]
        run, ev, sub = _ids(ref)
        n = len(run)
        out = {"Run": run, "Event": ev, "SubEvent": sub}
        ref_key = {k: i for i, k in enumerate(zip(run, ev, sub))}

        for tbl, cols in need.items():
            node = nodes.get(tbl)
            if node is None:
                for name, _ in cols:
                    out[name] = np.full(n, np.nan)
                continue

            r2, e2, s2 = _ids(node)
            same = (len(r2) == n and np.array_equal(r2, run)
                    and np.array_equal(e2, ev) and np.array_equal(s2, sub))
            if same:
                idx = None                       # hizali: dogrudan kullan
            else:
                idx = np.full(n, -1, dtype=np.int64)
                for j, k in enumerate(zip(r2, e2, s2)):
                    i = ref_key.get(k)
                    if i is not None:
                        idx[i] = j

            for name, col in cols:
                if col not in node.colnames:
                    out[name] = np.full(n, np.nan)
                    continue
                v = np.asarray(node.col(col), dtype=np.float64)
                if idx is None:
                    out[name] = v
                else:
                    a = np.full(n, np.nan)
                    ok = idx >= 0
                    a[ok] = v[idx[ok]]
                    out[name] = a
    return out


def load_sample(name, wanted, max_files=None):
    """Bir ornegin tum HDF5 dosyalarini oku ve birlestir."""
    pattern = SAMPLES[name]["hdf5"].replace(".hdf5", "*.hdf5")
    files = sorted(f for f in glob.glob(pattern) if "_smoke" not in f)
    if not files:
        files = sorted(glob.glob(pattern))
    if max_files:
        files = files[:max_files]
    if not files:
        print("[!] %s: HDF5 bulunamadi (%s)" % (name, pattern)); return None

    parts = [_load_one_file(f, wanted) for f in files]
    keys = parts[0].keys()
    data = {k: np.concatenate([p[k] for p in parts]) for k in keys}
    data["_n_files"] = len(files)
    print("%-8s %8d olay, %d dosya" % (name, len(data["Run"]), len(files)))
    return data


WANTED = sorted(set(NOISE_FEATURES) | set(MUON_FEATURES) | set(AUX))

data = {}
for name in SAMPLES:
    d = load_sample(name, WANTED, max_files=None)
    if d is not None:
        data[name] = d

### Sağlık kontrolü

Bir değişken bir örnekte **tamamen** NaN'sa, o kolon book edilmemiş demektir —
eğitime sokarsan model o değişkeni sessizce görmezden gelir.

In [ ]:
print("%-22s %s" % ("degisken", "  ".join("%8s" % s for s in data)))
for f in NOISE_FEATURES + MUON_FEATURES:
    row = []
    for s, d in data.items():
        v = d.get(f)
        frac = 100.0 * np.mean(~np.isfinite(v)) if v is not None else 100.0
        row.append("%7.1f%%" % frac)
    flag = ""
    if any(float(x.rstrip("%")) > 99.9 for x in row):
        flag = "  <-- HEP EKSIK"
    print("%-22s %s%s" % (f, "  ".join("%8s" % x for x in row), flag))
print("\n(NaN yuzdesi.  %100 olan bir kolon book edilmemis demektir.)")

## 5. Ağırlıklar

Üç kavram:

1. **Fiziksel ağırlık `w_phys` [Hz]** — dağılım ve kesim performansı için.
2. **Eğitim ağırlığı `w_train`** — teknik notun ön işlemesi: sinyal ve arka
   planın toplamları eşitlenir, sonra 0–1 aralığına çekilir
   (bölüm 3.6.1: *"event weights were manually re-scaled to 'balance' the
   samples... re-scaled to be in the range between 0 and 1"*).
3. **Ağırlıksız sayım** — istatistiksel yeterlilik.

Konvansiyon (`oscNext_L4_variables.py`'deki `PropagateGenieInfo` ile aynı):

```
GENIE : w = OneWeight * flux(E) / n_flux / n_files
        n_flux = I3GenieInfo.n_flux_events  (varsa)
               = NEvents * 0.7 (nu) / 0.3 (nubar)  (yoksa)
Noise : w = noise_weight * birim carpani / n_files   (pass3: 1/ns -> x1e9)
CORSIKA: simweights + GaisserH3a (yoksa yaklasik)
```

> `NORM`/`GAMMA` gerçek atmosferik akı **değil**, basit bir güç yasası. Mutlak
> oranlar teknik notun Tablo 13'üyle birebir tutmaz; şekil karşılaştırması ve
> eğitim için yeterlidir. Gerçek akı için `nuflux` (Honda) + salınım gerekir.

In [ ]:
NORM, GAMMA = 2e-2, -3.0
NU_FRAC, NUBAR_FRAC = 0.7, 0.3
NOISE_NS_SCALE = 1e9        # pass3: noise_weight 1/ns -> Hz.  pass2 ise 1.0


def genie_weight(d):
    E  = d["true_energy"]
    ow = d["OneWeight"]
    flux = NORM * np.power(E, GAMMA, where=E > 0, out=np.full_like(E, np.nan))

    n_flux = d.get("n_flux_events", np.full_like(E, np.nan)).copy()
    missing = ~np.isfinite(n_flux)
    if missing.any():
        frac = np.where(d["pdg"] < 0, NUBAR_FRAC, NU_FRAC)
        n_flux[missing] = (d["NEvents"] * frac)[missing]
        print("  [i] %d olayda n_flux_events yok -> NEvents * (%.1f/%.1f)"
              % (missing.sum(), NU_FRAC, NUBAR_FRAC))
    return ow * flux / n_flux / d["_n_files"]


def noise_weight(d):
    return d["noise_weight"] * NOISE_NS_SCALE / d["_n_files"]


def corsika_weight(d):
    """simweights yoksa kaba yaklasim -- mutlak oran guvenilmez."""
    print("  [!] CORSIKA agirligi yaklasik (simweights entegrasyonu yok). "
          "Mutlak oranlara guvenmeyin; sekil/egitim icin yeterli.")
    return np.ones(len(d["Run"])) / d["_n_files"]


WEIGHTERS = {"nue": genie_weight, "numu": genie_weight,
             "noise": noise_weight, "corsika": corsika_weight}

for name, d in data.items():
    print(name)
    w = np.asarray(WEIGHTERS[name](d), dtype=np.float64)
    w[~np.isfinite(w)] = 0.0
    d["w_phys"] = w
    tot = w.sum()
    print("  toplam oran = %.4e Hz,  maks/toplam = %.2f%%"
          % (tot, 100 * w.max() / tot if tot > 0 else np.nan))

### Ağırlık sağlık kontrolü

`maks/toplam > %5` ise tek bir olay oranı domine ediyor — istatistik yetersiz
ya da ağırlık hesabında sorun var.

v00.07 pass2'de L3 sonrası beklenen mertebeler (pass3'te birebir tutmaz):

| Bileşen | L3 oranı |
|---|---|
| νe CC | ~0.95 mHz |
| νμ CC | ~3.77 mHz |
| Atm. μ | ~505 mHz |
| Gürültü | ~36.6 mHz |

In [ ]:
n = len(data)
fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 2.8))
axes = np.atleast_1d(axes)
for ax, (name, d) in zip(axes, data.items()):
    w = d["w_phys"]; w = w[np.isfinite(w) & (w > 0)]
    if w.size == 0:
        ax.set_title("%s: agirlik yok" % name, fontsize=8); continue
    ax.hist(np.log10(w), bins=40, color="tab:blue")
    ax.set_title("%s\nmaks/toplam = %.1f%%" % (name, 100 * w.max() / w.sum()),
                 fontsize=8)
    ax.set_xlabel("log10(w_phys)", fontsize=7); ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()

## 6. pybdt DataSet'leri ve train/test ayrımı

`pybdt.ml.DataSet` bir *dict of numpy arrays*. Her BDT için dört dosya:
`sig_train`, `sig_test`, `bg_train`, `bg_test`.

**Eğitim ağırlığı** burada hesaplanıyor: sınıf toplamları eşitlenir, sonra
tüm ağırlıklar `[0, 1]` aralığına çekilir (teknik not, bölüm 3.6.1).

Ayrım **olay bazında rastgele** (%50/%50). Dosya bazında ayırmak daha
muhafazakâr olurdu ama örnek sayısı azken bu yeterli.

In [ ]:
TRAIN_FRAC = 0.5


def stack(samples, features):
    """Birden fazla ornegi tek bir {ad: dizi} sozluguna yigar."""
    out = {f: np.concatenate([data[s][f] for s in samples]) for f in features}
    out["w_phys"] = np.concatenate([data[s]["w_phys"] for s in samples])
    return out


def make_datasets(tag, sig_samples, bg_samples, features):
    """Bir BDT icin dort .ds dosyasi uret."""
    sig = stack(sig_samples, features)
    bg  = stack(bg_samples,  features)

    # --- egitim agirligi: sinif toplamlarini esitle, sonra [0,1]'e cek ---
    ws, wb = sig["w_phys"].copy(), bg["w_phys"].copy()
    for w in (ws, wb):
        w[~np.isfinite(w) | (w < 0)] = 0.0
    if ws.sum() > 0: ws /= ws.sum()
    if wb.sum() > 0: wb /= wb.sum()
    scale = max(ws.max(), wb.max())
    if scale > 0:
        ws /= scale; wb /= scale

    paths = {}
    for part, d, w in (("sig", sig, ws), ("bg", bg, wb)):
        n = len(w)
        istrain = rng.random(n) < TRAIN_FRAC
        for split, mask in (("train", istrain), ("test", ~istrain)):
            cols = {f: np.asarray(d[f], dtype=np.float64)[mask] for f in features}
            cols["weight"] = w[mask]
            cols["w_phys"] = np.asarray(d["w_phys"], dtype=np.float64)[mask]
            ds = ml.DataSet(cols)
            p = os.path.join(DS_BASE, "%s_%s_%s.ds" % (tag, part, split))
            util.save(ds, p)
            paths["%s_%s" % (part, split)] = p
            print("  %-18s %8d olay -> %s" % ("%s %s" % (part, split), mask.sum(),
                                              os.path.basename(p)))
    return paths


print("=== noise BDT ===")
DS_NOISE = make_datasets("L4_noise",
                         sig_samples=["nue", "numu"],
                         bg_samples=["noise"],
                         features=NOISE_FEATURES)

print("\n=== muon BDT ===")
DS_MUON = make_datasets("L4_muon",
                        sig_samples=["nue", "numu"],
                        bg_samples=["corsika"],
                        features=MUON_FEATURES)

## 7. Eğitim

`pybdt_train.py`'yi çağırır — eğitim mantığı tek yerde kalsın diye burada
tekrar yazılmıyor.

Hiperparametreler pybdt'nin kendi örneğinden (`resources/examples/
train_sample_bdt.sh`, IC79 νμ analizi). **oscNext için optimize edilmemiş** —
teknik notun Tablo 10'u LightGBM'e ait ve pybdt'ye taşınamaz. 8. bölümdeki
overtraining kontrolüne bakarak ayarla.

In [ ]:
HYPER = ["--num-trees", "300", "--depth", "3", "--beta", "0.7",
         "--prune-strength", "35", "--frac-random-events", "0.5",
         "--use-purity"]


def run_train(name, ds, features, extra=()):
    # --features ACIKCA veriliyor: .ds icinde w_phys gibi BDT girdisi
    # OLMAYAN kolonlar da var, otomatik secime birakilmamali.
    cmd = [sys.executable, TRAIN_PY, "--name", name, "--outdir", MODEL_DIR,
           "--sig-train", ds["sig_train"], "--bg-train", ds["bg_train"],
           "--sig-test",  ds["sig_test"],  "--bg-test",  ds["bg_test"],
           "--features", ",".join(features),
           ] + HYPER + list(extra)
    print("$ " + " ".join(shlex.quote(c) for c in cmd) + "\n")
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout)
    if p.returncode != 0:
        print("--- HATA ---\n" + (p.stderr or "")[-3000:])
    return p.returncode == 0


run_train("L4_noise", DS_NOISE, NOISE_FEATURES)

In [ ]:
run_train("L4_muon", DS_MUON, MUON_FEATURES)

## 8. Doğrulama

`pybdt_train.py` `.validator` dosyasını kaydetti — skorlar zaten hesaplanmış
durumda, buradan interaktif olarak inceleyebiliriz.

**Overtraining ölçütü**: pybdt'nin KS testi. `p_KS ≲ 0.01` ise overtraining
var demektir (`pybdt/resources/docs/man_overtraining.rst`) → ağaç derinliğini
düşür, `prune-strength`'i artır, ya da `min-split`'i büyüt.

In [ ]:
from pybdt.validate import Validator

def load_validator(name):
    p = os.path.join(MODEL_DIR, "%s.validator" % name)
    meta = json.load(open(os.path.join(MODEL_DIR, "%s.json" % name)))
    return util.load(p), meta


V, META = {}, {}
for name in ("L4_noise", "L4_muon"):
    try:
        V[name], META[name] = load_validator(name)
        m = META[name]["metrics"]
        print("%-10s %d agac,  p_KS sinyal=%.4f  arkaplan=%.4f"
              % (name, META[name]["n_trees"], m["ks_signal"], m["ks_background"]))
    except FileNotFoundError:
        print("%-10s henuz egitilmedi" % name)

In [ ]:
from IPython.display import Image, display

for name in V:
    for kind in ("overtrain", "dist", "rate"):
        p = os.path.join(MODEL_DIR, "%s_%s.png" % (name, kind))
        if os.path.exists(p):
            print(name, kind); display(Image(filename=p))

## 9. Kesim seçimi

pybdt skoru LightGBM'in `P(sinyal)` olasılığıyla **aynı ölçekte değil** —
teknik nottaki 0.70 / 0.65 değerleri buraya taşınamaz. Kesimi kendi
rate-vs-cut eğrinden seç.

Referans hedefler (v00.07, pass2):
- **noise**: 36.6 mHz → <0.3 mHz gürültü, nötrinoların ~%96'sı korunur
- **muon**: muonların %94'ü atılır, nötrinoların %87'si korunur

In [ ]:
def scan_cut(name, n=200):
    """Kesim degerine karsi sinyal verimi / arkaplan reddi (test seti)."""
    v, meta = V[name], META[name]
    expr = meta["score_expr"]

    # skorlar; agirlik olarak FIZIKSEL agirligi kullan (egitim agirligi
    # siniflari esitlemek icin olceklenmisti, fiziksel oran degil)
    s = v.eval("test_sig", expr)
    b = v.eval("test_bg",  expr)
    try:
        ws = v.eval("test_sig", "w_phys")
        wb = v.eval("test_bg",  "w_phys")
    except Exception:
        print("  [!] w_phys yok -> egitim agirligi kullaniliyor, "
              "oranlar fiziksel DEGIL")
        ws = v.get_values_weights("test_sig", expr)[1]
        wb = v.get_values_weights("test_bg",  expr)[1]
    ws = np.where(np.isfinite(ws), ws, 0.0)
    wb = np.where(np.isfinite(wb), wb, 0.0)
    lo = min(s.min(), b.min()); hi = max(s.max(), b.max())
    cuts = np.linspace(lo, hi, n)

    eff = np.array([ws[s >= c].sum() for c in cuts]) / ws.sum()
    rej = 1 - np.array([wb[b >= c].sum() for c in cuts]) / wb.sum()

    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.plot(cuts, 100 * eff, label="sinyal verimi", color="tab:blue")
    ax.plot(cuts, 100 * rej, label="arkaplan reddi", color="tab:red")
    ax.set_xlabel("pybdt skoru"); ax.set_ylabel("%"); ax.grid(alpha=.3)
    ax.legend(fontsize=8); ax.set_title(name, fontsize=10)
    plt.tight_layout(); plt.show()
    return cuts, eff, rej


for name in V:
    cuts, eff, rej = scan_cut(name)
    # ornek: arkaplanin %94'unu atan kesim nerede?
    i = np.argmin(np.abs(rej - 0.94))
    print("%s: red %%94 -> kesim %.3f,  sinyal verimi %%%.1f"
          % (name, cuts[i], 100 * eff[i]))

## 10. Modeli frame'e uygulama

Eğitim bittikten sonra `.i3` dosyalarını yeniden işleyip her frame'e BDT
skorunu yazmak için `pybdt_classifier_module.py` kullanılır:

```python
from pybdt_classifier_module import PyBDTClassifier

tray.Add(PyBDTClassifier, "noise_clf",
         ModelFile="<MODEL_DIR>/L4_noise.bdt",
         OutputKey="L4_NoiseClassifier_pybdt")

tray.Add(PyBDTClassifier, "muon_clf",
         ModelFile="<MODEL_DIR>/L4_muon.bdt",
         OutputKey="L4_MuonClassifier_pybdt")
```

> **Dikkat:** Modül, değişkenleri frame'den `l4_classifier_module.py`'deki
> `FEATURE_MAP` üzerinden okur. 3. bölümdeki `REGISTRY` ile aynı değişkenleri
> göstermeli — biri HDF5 kolonunu, diğeri frame anahtarını tarif eder. İkisi
> ayrışırsa eğitimde bir şey, uygulamada başka bir şey okunur ve model
> **hata vermeden** yanlış sonuç üretir.

In [ ]:
# Registry <-> FEATURE_MAP tutarlilik kontrolu (icetray gerektirmez -- AST ile)
import ast

def feature_map_names(path="l4_classifier_module.py"):
    if not os.path.exists(path):
        return None
    tree = ast.parse(open(path).read())
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            for t in node.targets:
                if isinstance(t, ast.Name) and t.id == "FEATURE_MAP":
                    return {k.value for k in node.value.keys}
    return None


fm = feature_map_names(os.path.join(L4_CODE_DIR, "l4_classifier_module.py"))
if fm is None:
    print("[!] l4_classifier_module.py okunamadi")
else:
    used = set(NOISE_FEATURES) | set(MUON_FEATURES)
    missing = sorted(used - fm)
    if missing:
        print("[!] FEATURE_MAP'te TANIMSIZ (uygulama asamasinda patlar):")
        for m in missing:
            print("      ", m)
    else:
        print("OK -- egitimde kullanilan %d degiskenin hepsi FEATURE_MAP'te var"
              % len(used))

## Kontrol listesi

- [ ] Smoke test temiz, tablolar book ediliyor (bölüm 1–2)
- [ ] Registry'deki tüm kolonlar HDF5'te bulundu (bölüm 3)
- [ ] Hiçbir değişken %100 NaN değil (bölüm 4)
- [ ] Ağırlıklarda tek olay domine etmiyor, mertebeler makul (bölüm 5)
- [ ] `p_KS > 0.01` — overtraining yok (bölüm 8)
- [ ] Kesim seçildi, verim/red referans mertebelerde (bölüm 9)
- [ ] `REGISTRY` ↔ `FEATURE_MAP` tutarlı (bölüm 10)

**Hâlâ doğrulanmamış olanlar** (bkz. `CLAUDE.md` → *Açık riskler*): VICH,
`accumulated_time`, `separation_in_cogs` saf Python'la yeniden yazıldı ve
orijinal C++ ile karşılaştırılmadı. `FullTimeLengthRatio`'nun yönü ve
ağırlık zincirinin mutlak ölçeği de doğrulanmadı.